In [1]:
# Local packages
from jupyter_option import args
import utility
import data 
import model

import torch
import numpy as np
from importlib import reload
import matplotlib.pyplot as plt

print_args = False

args.ext = 'img'
args.cpu = True
args.model = 'TOY'

args.dir_data = '../../Data'
args.data_range = '1-10/15-20'
#args.data_train = ['DIV2K_TIFF']
#args.data_test = ['DIV2K_TIFF']
args.data_train = ['DIV2K']
args.data_test =  ['DIV2K']
args.load = 'toy_x2'
args.resume = -1
args.n_resblocks = 32 
args.n_feats = 32 
args.scale = [2]


args.n_colors = 3
args.test_only = True

checkpoint = utility.checkpoint(args)

def _permute(tensor, ax1, ax2, ax3):
    if tensor.dim() != 3:
        raise RuntimeError("Esperava tensor 3D mas recebeu tensor de formato {}".format(tensor.shape))
    return tensor.permute(ax1,ax2,ax3)
    
def channel_first(tensor):
    return _permute(tensor, 2, 0, 1)

def channel_last(tensor):
    return _permute(tensor, 1, 2, 0)

if print_args:
    for arg in vars(args):
        print(arg, getattr(args,arg))

%matplotlib qt5

Continue from epoch 9...


### Testa modelo com dataset de teste

In [2]:
# Cria dataset e instancia modelo
print_dataset = False

data_loader = data.Data(args)
loader_test = data_loader.loader_test[0]
iter_loader = iter(loader_test)

print("Tamanho loader_test:", len(loader_test))
if print_dataset:
    for lr, hr, img_idx in loader_test:
        print(img_idx, lr.shape, hr.shape)
        
m = model.Model(args, checkpoint)

Tamanho loader_test: 6
Making model...


In [25]:
# Carrega amostra de teste
def prepare(*fargs):
    device = torch.device('cpu' if args.cpu else 'cuda')
    def _prepare(tensor):
        if args.precision == 'half': tensor = tensor.half()
        return tensor.to(device)
    return [_prepare(a) for a in fargs]

lr, hr, filename = iter_loader.__next__()
lr, hr = prepare(lr, hr)

print("Escala:", args.scale[0])
print("Imagem:", filename)
print("LR:", lr.shape)
print("LR:", hr.shape)

Escala: 2
Imagem: ('0810',)
LR: torch.Size([1, 3, 768, 1020])
LR: torch.Size([1, 3, 1536, 2040])


In [26]:
# Inferência do modelo (SR)
m.eval()

sr = m(lr, args.scale[0])
print("Imagem SR:", sr.min().item(), sr.max().item(), sr.mean().item(), sr.std().item())

# Computa PSNR
sr = utility.quantize(sr, args.rgb_range)
psnr = utility.calc_psnr(sr, hr, args.scale[0], args.rgb_range, dataset=loader_test)

print("PSNR:", format(psnr, '.3f'))

Imagem SR: -8.877135276794434 251.4644012451172 110.70038604736328 43.84208679199219
PSNR: 30.109


In [28]:
# Plota SR e HR
interp = 'nearest'
plot_hr = True

sr_img = channel_last(sr.detach().squeeze().byte())
plt.figure()
plt.imshow(sr_img, interpolation=interp)
plt.title('SR')

if plot_hr:
    hr_img = channel_last(hr.detach().squeeze().byte())
    plt.figure()
    plt.imshow(hr_img, interpolation=interp)
    plt.title('HR')

plt.show()

In [27]:
# Plota DIFF
color = 'inferno'

diff = sr - hr
diff = diff.squeeze().abs().mean(dim=0).detach()
#diff = channel_last(diff.detach().squeeze())
vmax = diff.abs().max().item()
print("Max:", vmax)
print("Mean:", diff.mean().item())
#vmin = -vmax
vmin = 0

plt.figure()
plt.imshow(diff, vmin=vmin, vmax=vmax, cmap=color)
plt.title('Diff')
plt.colorbar().set_label('pixel distance')
plt.show()

Max: 68.33333587646484
Mean: 4.776219844818115
